# langgraph 核心概念

## 安装

In [2]:
!uv add langgraph=="1.2.10"

Resolved 40 packages in 42ms                                         
   Building langgraph-python @ file:///Users/chenjie/Desktop/FutureArchitect/%E5
      Built langgraph-python @ file:///Users/chenjie/Desktop/FutureArchitect/%E5
⠹ Preparing packages... (1/21)                                                  
⠹ Preparing packages... (1/21)------------------     0 B/18.57 KiB           
⠹ Preparing packages... (1/21)------------------ 8.00 KiB/18.57 KiB          
⠹ Preparing packages... (1/21)--------- 16.00 KiB/18.57 KiB         
⠹ Preparing packages... (1/21)--------- 18.57 KiB/18.57 KiB         
⠹ Preparing packages... (1/21)/21)                                                       
⠹ Preparing packages... (1/21)------------------     0 B/31.55 KiB           
⠹ Preparing packages... (1/21)------------------ 8.00 KiB/31.55 KiB          
⠹ Preparing packages... (1/21)--------------- 16.00 KiB/31.55 KiB         
⠹ Preparing packages... (1/21)--------- 24.00 KiB/31.55 KiB         
⠹ P

## 第一个 langgraph 程序

In [3]:
from langgraph_python.graph.my_first_graph import graph
graph.invoke({})

{'node1': 'node2 result', 'node2': 'node2 result'}

Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a190b2-4da2-714b-bf1c-a8ed41dc812c,id=01a190b2-4da2-714b-bf1c-a8ed41dc812c; trace=01a190b2-4da2-714b-bf1c-a8ed41dc812c,id=01a190b2-4da8-7521-be6c-9acc1cbbec73; trace=01a190b2-4da2-714b-bf1c-a8ed41dc812c,id=01a190b2-4da8-7521-be6c-9acc1cbbec73; trace=01a190b2-4da2-714b-bf1c-a8ed41dc812c,id=01a190b2-4daa-769b-a2ac-c9e7e63a8168; trace=01a190b2-4da2-714b-bf1c-a8ed41dc812c,id=01a190b2-4daa-769b-a2ac-c9e7e63a8168; trace=01a190b2-4da2-714b-bf1c-a8ed41dc812c,id=01a190b2-4da2-714b-bf1c-a8ed41dc812c


In [4]:
from langchain_core.runnables import Runnable
print(isinstance(graph, Runnable))

True


## 核心概念

<img src="./assets/图.svg">

### 三个阶段

定义 -> 编译 -> 运行

1. **定义**

   - 节点
   - 边
   - 图状态结构

2. **编译**

   定义好的图是不可运行的，它只是一个描述。

   若要运行图，需要进行编译

   编译好的图，节点和边会固定下来，不能动态变化。

3. **运行**

   编译结果可以运行。

   > langgraph 底层使用 Pregel 系统运行图
   >
   > Pregel来自2010年的谷歌论文，论文中详细阐述了图的执行。
   >
   > https://dl.acm.org/doi/10.1145/1807167.1807184

### 节点

节点可以接收以下类型：
- 可调用对象：实现了`__call__`的对象
- Runnable对象：`Runnable`是`langchain_core`中定义的类型

In [ ]:
from langchain_core.runnables import Runnable

print(hasattr(Runnable, "invoke"))      # 执行
print(hasattr(Runnable, "ainvoke"))     # 异步执行
print(hasattr(Runnable, "stream"))      # 流式执行
print(hasattr(Runnable, "astream"))     # 异步流式执行
print(hasattr(Runnable, "batch"))       # 批量执行
print(hasattr(Runnable, "abatch"))      # 异步批量执行

In [ ]:
class A:
    pass

abc = A()
print(abc._)

AttributeError: 'A' object has no attribute '__bases__'

### 图状态

图状态是整张图的共享数据空间，图在执行期间，每个节点都可以获取到完整的图状态，也可以更新图状态。

图执行完后，得到的结果就是最新的图状态。

<img src="./assets/节点.svg">

图状态本质上是一个字典，字典的每一个字段称之为 channel （通道）